# 🔍 Notebook 01: Initial Data Ingestion & Quality Audit

**Project:** Customer Churn & Lifetime Value (LTV) Engine  
**Author:** Adarsh Kumar Pandey (*Data Analyst*)  
**Dataset:** `data/WA_Fn-UseC_-Telco-Customer-Churn.csv` (IBM Watson Telco Cohort)  
**Objective:** Audit raw subscriber records, verify schema integrity, detect missing or hidden whitespace characters, inspect data types, and check for duplicate customer entries.

## 1. Load Baseline Raw Dataset

In [1]:
import os
import numpy as np
import pandas as pd

# Load raw Telco customer churn dataset
data_path = '../data/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(data_path)

print(f'Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

Dataset Shape: 7043 rows, 21 columns


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Inspect Column Types and Non-Null Counts

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

## 3. Explicit Missing Values Check

In [3]:
missing_counts = df.isnull().sum()
print('Missing values per column:')
print(missing_counts[missing_counts > 0] if missing_counts.sum() > 0 else 'No explicit NaN values detected.')

Missing values per column:
No explicit NaN values detected.


## 4. Audit Hidden Whitespace Characters in `TotalCharges`
In telecom billing records, `TotalCharges` is often parsed as `object`/string because unbilled new accounts contain whitespace (`' '`) rather than `NaN`.

In [4]:
# Identify whitespace rows in TotalCharges
whitespace_mask = df['TotalCharges'].str.strip() == ''
blank_records = df[whitespace_mask]

print(f'Total whitespace records found in TotalCharges: {len(blank_records)}')
blank_records[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Contract', 'Churn']]

Total whitespace records found in TotalCharges: 11


,customerID,tenure,MonthlyCharges,TotalCharges,Contract,Churn
488,4472-LVYGI,0,52.55,,Two year,No
753,3115-CZMZD,0,20.25,,Two year,No
936,5709-LVOEQ,0,80.85,,Two year,No
1082,4367-NUYAO,0,25.75,,Two year,No
1340,1371-DWPAZ,0,56.05,,Two year,No
3331,7644-OMVMY,0,19.85,,Two year,No
3826,3213-VVOLG,0,25.35,,Two year,No
4380,2520-SGTTA,0,20.00,,Two year,No
5218,2923-ARZLG,0,19.70,,One year,No
6670,4075-WKNIU,0,73.35,,Two year,No


> **Key Audit Finding:** All 11 whitespace records correspond to customers with `tenure = 0`. These are newly onboarded subscribers within their first billing cycle who have not yet received an invoice.

## 5. Duplicate Records & Unique Identifier Audit

In [5]:
duplicate_ids = df['customerID'].duplicated().sum()
total_records = len(df)
print(f'Total Records: {total_records}')
print(f'Duplicate Customer IDs: {duplicate_ids}')
print(f'ID Uniqueness: {(total_records - duplicate_ids) / total_records * 100:.2f}%')

Total Records: 7043
Duplicate Customer IDs: 0
ID Uniqueness: 100.00%


## 6. Audit Summary & Next Steps

| Audit Metric | Finding | Action Required |
| :--- | :--- | :--- |
| **Total Rows & Columns** | 7,043 rows, 21 columns | Baseline confirmed |
| **TotalCharges Formatting** | Stored as string with 11 blank spaces | Cast to float, impute 0.00 for `tenure = 0` |
| **Customer ID Uniqueness** | 100% unique (0 duplicates) | Identifier verified |
| **Target Variable (`Churn`)** | Complete (5,174 No, 1,869 Yes) | Ready for cleaning and EDA |